# UTS Graf Sains Data — Sistem Rekomendasi dengan Neo4j (RetailRocket)

| | |
|---|---|
| **Nama** | Zulaika Fadila Handini |
| **NIM** | 232031554182 |
| **No. urut (daftar GSD 2023E)** | 24 |
| **Mata kuliah** | Graf Sains Data (3 SKS), Gasal 2026/2027 |
| **Dataset** | RetailRocket e-commerce (`events.csv`, sumber: Roman Zykov, Kaggle) |
| **Fold training (P1)** | 24 Mei 2015 – 12 Agustus 2015 |
| **Fold testing (P2)** | 13 Agustus 2015 – 2 September 2015 |

In [ ]:
# Impor semua pustaka yang dipakai di notebook ini
import os, sys, time, getpass

import numpy as np                      # array / matriks
import pandas as pd                     # tabel data (DataFrame)
import scipy.sparse as sp               # matriks sparse (CSR) untuk VSM
import networkx as nx                   # menggambar graf bipartit
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm import tqdm                   # progress bar impor batch
from sklearn.preprocessing import normalize   # normalisasi L2 untuk cosine similarity

import neo4j
from neo4j import GraphDatabase, RoutingControl

print("Python :", sys.executable)
print("neo4j  :", neo4j.__version__)

In [ ]:
# Kredensial koneksi Neo4j.
# Password TIDAK ditulis di notebook agar aman saat diunggah ke GitHub:
# ketik saat diminta, atau set variabel lingkungan NEO4J_PASSWORD.
URI = os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687")
USER = os.environ.get("NEO4J_USER", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD") or getpass.getpass("Password Neo4j: ")
AUTH = (USER, PASSWORD)

In [26]:
# Tes koneksi ke server Neo4j (error jika server mati / password salah)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Terhubung ke Neo4j!")

Terhubung ke Neo4j!


In [27]:
# Kosongkan database secara bertahap (aman untuk data besar)
def kosongkan_database(batch=10000):
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database="neo4j") as session:
            session.run(
                f"MATCH (n) CALL (n) {{ DETACH DELETE n }} IN TRANSACTIONS OF {int(batch)} ROWS"
            ).consume()
    print("Database dikosongkan.")

## Tahap Tutorial penggunaan driver Neo4j

contoh graf pertemanan: membuat node `Person` beserta relasi `KNOWS` (`add_friend`), lalu membaca kembali teman seseorang (`print_friends`).

Poin penting pada kode:
- `MERGE` membuat node/relasi **hanya jika belum ada**, sehingga aman dijalankan berulang kali (tidak terduplikasi).
- `driver.execute_query(...)` mengirim query Cypher beserta parameternya (`$name`, `$friend_name`) ke server.
- `database_="neo4j"` menentukan database tujuan, dan `routing_=RoutingControl.READ` menandakan query bersifat baca saja.

In [28]:
# Fungsi menambah node Person dan relasi KNOWS (a)-[:KNOWS]->(friend)
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

In [29]:
# Fungsi mencetak semua teman dari seorang Person (diurutkan nama)
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

In [30]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


### Jawaban Pertanyaan Pendahuluan (Checkpoint)

1. **Nama database yang digunakan:** neo4j, terlihat dari parameter database_="neo4j" pada execute_query.
2. **Akses database lain dalam 1 instance:** ganti nilai database_ pada execute_query atau gunakan driver.session(database="namadb"). Pada Cypher, SHOW DATABASES menampilkan daftar database dan :USE namadb berpindah database (di Neo4j Browser). Membuat database baru dengan CREATE DATABASE hanya tersedia pada edisi Enterprise atau Aura berbayar.
3. **Jenis graf pertemanan:** graf **berarah** (*directed graph*) dengan node berlabel Person dan relasi KNOWS. Bentuknya *star* dengan Arthur sebagai pusat yang mengarah ke Guinevere, Lancelot, dan Merlin. Graf ini tidak berbobot.

In [ ]:
# "CYPHER Total Reset": kosongkan database agar data tutorial tidak bercampur dengan data RetailRocket
# (dihapus bertahap agar tidak kehabisan memori)
kosongkan_database()

## Dataset RetailRocket

Dataset diunduh dari Kaggle (`retailrocket/ecommerce-dataset`) dan `events.csv` diletakkan di folder `data/`. Kolom pada `events.csv`:

In [32]:
# Muat events.csv dan lihat bentuk datanya
import pandas as pd

df = pd.read_csv("data/events.csv")
print(df.shape)
df.head()

(2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [33]:
# Ubah timestamp (ms) menjadi tanggal agar rentang waktu data terbaca
print(df.shape)
print(df.dtypes)

df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms")

print("Tanggal awal :", df["datetime"].min())
print("Tanggal akhir:", df["datetime"].max())

(2756101, 5)
timestamp          int64
visitorid          int64
event                str
itemid             int64
transactionid    float64
dtype: object
Tanggal awal : 2015-05-03 03:00:04.384000
Tanggal akhir: 2015-09-18 02:59:47.788000


In [34]:
# Komposisi jenis event pada seluruh data
df["event"].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [35]:
# Jumlah visitor dan item unik pada seluruh data
print("Visitor unik:", df["visitorid"].nunique())
print("Item unik   :", df["itemid"].nunique())

Visitor unik: 1407580
Item unik   : 235061


**Ringkasan eksplorasi awal.** Data berisi 2.756.101 baris interaksi dari 1.407.580 visitor dan 235.061 item, dengan rentang waktu 3 Mei 2015 sampai 18 September 2015 (UTC). Event `view` mendominasi (2.664.312 baris, sekitar 96,7%), sedangkan `addtocart` 69.332 dan `transaction` 22.457. Rentang ini memuat kedua periode filter yang diminta.

## Wrangling dan EDA

### No. 1 Penentuan dua periode waktu (filter)
| Fold | Awal | Akhir |
|---|---|---|
| **Training (P1)** | 24 Mei 2015 | 12 Agustus 2015 |
| **Testing (P2)** | 13 Agustus 2015 | 2 September 2015 |


In [36]:
# Definisi periode dan fungsi filter berdasarkan rentang tanggal
P1_START, P1_END = "2015-05-24", "2015-08-12"   # training
P2_START, P2_END = "2015-08-13", "2015-09-02"   # testing

def filter_period(data, start, end):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end) + pd.Timedelta(days=1)   # tanggal akhir ikut (inklusif)
    return data[(data["datetime"] >= s) & (data["datetime"] < e)]

In [37]:
# Menerapkan filter, lalu cek jumlah baris, rentang waktu, dan komposisi event
df_p1 = filter_period(df, P1_START, P1_END)
df_p2 = filter_period(df, P2_START, P2_END)

for nama, d in [("P1 (training)", df_p1), ("P2 (testing)", df_p2)]:
    print(nama, "->", d.shape[0], "baris")
    print("  awal :", d["datetime"].min())
    print("  akhir:", d["datetime"].max())
    print(d["event"].value_counts().to_dict())

P1 (training) -> 1683395 baris
  awal : 2015-05-24 00:00:04.950000
  akhir: 2015-08-12 23:59:59.400000
{'view': 1627535, 'addtocart': 42062, 'transaction': 13798}
P2 (testing) -> 373128 baris
  awal : 2015-08-13 00:00:05.707000
  akhir: 2015-09-02 23:59:54.456000
{'view': 359982, 'addtocart': 9925, 'transaction': 3221}


**Hasil filter.** P1 (training) berisi **1.683.395** baris (`view` 1.627.535, `addtocart` 42.062, `transaction` 13.798) dan P2 (testing) berisi **373.128** baris (`view` 359.982, `addtocart` 9.925, `transaction` 3.221). Batas waktunya tepat: P1 berakhir 12 Agustus pukul 23:59 dan P2 dimulai 13 Agustus pukul 00:00.

### No. 2 Menyimpan hasil filter ke file `.csv`

Hasil filter disimpan menjadi dua file dengan kolom yang sama dengan `events.csv` asli (kolom bantu `datetime` tidak ikut disimpan), sehingga kode impor dari soal tetap dapat dipakai.

In [38]:
# Sel 2.1.2 - Simpan hasil filter P1 dan P2 ke file CSV
kolom = ["timestamp", "visitorid", "event", "itemid", "transactionid"]

df_p1[kolom].to_csv("data/events_p1_train.csv", index=False)
df_p2[kolom].to_csv("data/events_p2_test.csv", index=False)

In [39]:
#Constraint unik untuk Visitor dan Item (juga membuat indeks untuk MERGE)
def create_constraints(session):
    session.run(
        "CREATE CONSTRAINT visitor_id IF NOT EXISTS "
        "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE"
    )
    session.run(
        "CREATE CONSTRAINT item_id IF NOT EXISTS "
        "FOR (i:Item) REQUIRE i.itemId IS UNIQUE"
    )

In [ ]:
# Query Cypher impor satu batch.
# UNWIND memecah daftar baris; MERGE membuat Visitor/Item sekali saja;
# CALL (v, i, row) { ... UNION ALL ... } memilih jenis relasi sesuai nilai event.
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL (v, i, row) {
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [ ]:
# Impor penuh P1 secara batch (+- 3,5 menit; 169 batch).
# Database dikosongkan dulu (bertahap) karena relasi dibuat dengan CREATE
# (impor ulang akan menggandakan relasi).
# Set RUN_IMPORT = False jika data P1 sudah ada di Neo4j dan ingin melewati sel ini.
RUN_IMPORT = True

csv_file_path = "data/events_p1_train.csv"
batch_size = 10000
total_rows = 1683395                         # jumlah baris P1 (lihat output sel filter P1/P2 di atas)
total_chunks = (total_rows // batch_size) + 1

if RUN_IMPORT:
    kosongkan_database()                     # menggantikan MATCH (n) DETACH DELETE n

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()

        with driver.session(database="neo4j") as session:
            create_constraints(session)

            chunks = pd.read_csv(csv_file_path, chunksize=batch_size,
                                 dtype={"transactionid": "string"})   # ID transaksi sebagai teks

            for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
                chunk["transactionid"] = chunk["transactionid"].fillna("")   # NaN -> string kosong
                session.execute_write(import_batch, chunk.to_dict(orient="records"))

    print("Proses impor selesai.")

In [ ]:
# Verifikasi hasil impor: jumlah simpul per label dan relasi per jenis
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database="neo4j") as session:
        print(session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS jumlah").data())
        print(session.run("MATCH ()-[r]->() RETURN type(r) AS tipe, count(*) AS jumlah").data())

### No. 3 Komposisi Subgraph (Fold Training P1: 24 Mei 2015 – 12 Agustus 2015)

Subgraph dibangun dari data interaksi RetailRocket pada periode P1 (1.683.395 baris), lalu diimpor ke Neo4j. Komposisinya diperoleh dengan menjalankan query Cypher melalui Python API (`driver.execute_query`).

### query Cypher

**1. Menghitung simpul per jenis**

```cypher
MATCH (n)
RETURN labels(n)[0] AS jenis_simpul, count(n) AS jumlah
ORDER BY jumlah DESC
```

- `MATCH (n)` mencocokkan semua simpul di database. Dan `n` adalah variabel yang mewakili setiap simpul.
- `labels(n)` mengembalikan daftar label dari sebuah simpul. Di sini hanya ada satu label (`Visitor` atau `Item`), sehingga `[0]` mengambil label tersebut.
- `count(n)` adalah fungsi agregasi yang menghitung simpul. Karena `labels(n)[0]` ikut ditampilkan di `RETURN`, Cypher otomatis mengelompokkan hasil per label. Ini mirip `GROUP BY` pada SQL, tetapi di Cypher tidak perlu ditulis secara eksplisit.
- `ORDER BY jumlah DESC` mengurutkan hasil dari jumlah terbesar ke terkecil.

**2. Menghitung relasi per jenis**

```cypher
MATCH ()-[r]->()
RETURN type(r) AS jenis_relasi, count(r) AS jumlah
ORDER BY jumlah DESC
```

- `MATCH ()-[r]->()` mencocokkan semua relasi berarah. Simpul di kiri dan kanan dikosongkan `()` karena yang dihitung hanya relasinya. `r` adalah variabel untuk relasi.
- `type(r)` mengembalikan nama jenis relasi (`VIEWED`, `ADDED_TO_CART`, atau `BOUGHT`).
- `count(r)` menghitung relasi, dikelompokkan per jenis seperti pada query simpul.

**Pemanggilan melalui Python API**

Query dikirim ke server Neo4j dengan `driver.execute_query(query, database_="neo4j", routing_=RoutingControl.READ)`. Parameter `routing_=RoutingControl.READ` menandakan bahwa query bersifat baca saja. Hasilnya berupa daftar `records`. Setiap record diubah menjadi dictionary dengan `.data()`, lalu dimasukkan ke DataFrame pandas supaya mudah ditampilkan sebagai tabel.

In [ ]:
q_simpul = """
MATCH (n)
RETURN labels(n)[0] AS jenis_simpul, count(n) AS jumlah
ORDER BY jumlah DESC
"""

q_relasi = """
MATCH ()-[r]->()
RETURN type(r) AS jenis_relasi, count(r) AS jumlah
ORDER BY jumlah DESC
"""

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    rec_s, _, _ = driver.execute_query(q_simpul, database_="neo4j", routing_=RoutingControl.READ)
    rec_r, _, _ = driver.execute_query(q_relasi, database_="neo4j", routing_=RoutingControl.READ)

df_simpul = pd.DataFrame([r.data() for r in rec_s])
df_relasi = pd.DataFrame([r.data() for r in rec_r])

display(df_simpul)
display(df_relasi)

**Komposisi simpul**

| Jenis simpul | Jumlah |
|---|---|
| Visitor | 870.345 |
| Item | 190.242 |
| **Total** | **1.060.587** |

**Komposisi relasi**

| Jenis relasi | Jumlah | Persentase |
|---|---|---|
| VIEWED | 1.627.535 | 96,68% |
| ADDED_TO_CART | 42.062 | 2,50% |
| BOUGHT | 13.798 | 0,82% |
| **Total** | **1.683.395** | 100% |

Jumlah relasi sama dengan jumlah baris data P1, jadi tidak ada baris yang hilang atau terduplikasi saat impor.
### Interpretasi

1. Struktur graf bipartit. Semua relasi berasal dari simpul `Visitor`  menuju simpul `Item`. Tidak ada relasi antar-Visitor maupun antar-Item. Kedua jenis simpul ini nantinya menjadi dasar proyeksi graf (VSM) pada no. 4 dan 5.
2. Visitor lebih banyak daripada Item Rasionya sekitar 4,6 visitor untuk setiap item.
3. Graf sangat jarang (sparse). Rata-rata satu visitor hanya memiliki sekitar 1,9 relasi, sedangkan satu item memiliki sekitar 8,8 relasi. Kepadatan graf hanya sekitar 1,0 × 10⁻⁵ dari semua kemungkinan pasangan Visitor–Item.
4. Data didominasi event `view` Sekitar 96,7% relasi adalah `VIEWED`, sedangkan `ADDED_TO_CART` dan `BOUGHT` hanya 2,5% dan 0,8%. Interaksi yang menunjukkan minat kuat (beli) sangat sedikit, sehingga ini perlu diperhatikan pada tahap rekomendasi dan evaluasi (Hit@5 dan NDCG@5).

### No. 4 — Vector Space Modelling (VSM) simpul User

Pada VSM, setiap **Visitor** (User) dipandang sebagai satu **vektor** pada ruang berdimensi sebanyak jumlah **Item**: tiap dimensi mewakili satu item, dan nilainya adalah skor interaksi user terhadap item tersebut. Gabungan semua vektor membentuk matriks **User × Item** (`M_user`).

Skor interaksi memakai bobot yang mencerminkan kedekatan terhadap pembelian: `VIEWED` = 1, `ADDED_TO_CART` = 3, `BOUGHT` = 5. Interaksi berulang pada pasangan user–item yang sama **dijumlahkan** (agregasi dilakukan Cypher lewat `sum(CASE type(r) ...)`).

Karena ukurannya sangat besar, matriks disimpan sebagai **sparse matrix (CSR)** dari `scipy`; bentuk dense tidak mungkin (lihat perbandingan memori pada output).

In [ ]:
# Tarik skor interaksi (Visitor, Item) dari Neo4j dan bangun matriks sparse User x Item
import numpy as np
import scipy.sparse as sp

q_ui = """
MATCH (v:Visitor)-[r]->(i:Item)
RETURN v.visitorId AS u, i.itemId AS i,
       sum(CASE type(r)
             WHEN 'VIEWED' THEN 1
             WHEN 'ADDED_TO_CART' THEN 3
             WHEN 'BOUGHT' THEN 5
           END) AS skor
"""

us, its, vals = [], [], []
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database="neo4j") as session:
        for rec in session.run(q_ui):      # di-stream, tidak ditampung sekaligus di server
            us.append(rec["u"]); its.append(rec["i"]); vals.append(rec["skor"])

print("Pasangan (user, item) unik:", len(vals))

# Ubah ID asli menjadi indeks baris/kolom 0..n-1.
# sort=True: indeks diurutkan menurut ID, sehingga tidak bergantung pada urutan baris
# yang dikembalikan Neo4j (supaya sampel acak, plot, dan hasil bisa direproduksi).
u_idx, user_ids = pd.factorize(pd.Series(us), sort=True)
i_idx, item_ids = pd.factorize(pd.Series(its), sort=True)

M_user = sp.csr_matrix(
    (np.array(vals, dtype=np.float32), (u_idx, i_idx)),
    shape=(len(user_ids), len(item_ids)),
)

In [ ]:
# Karakteristik vektor/matriks User: dimensi, shape, sparsity, memori, distribusi
n_user, n_item = M_user.shape
nnz = M_user.nnz

print("Shape matriks User x Item :", M_user.shape)
print("Dimensi vektor user       :", n_item, "(satu dimensi per item)")
print("Jumlah elemen non-nol     :", nnz)
print("Kepadatan (density)       :", nnz / (n_user * n_item))
print("Sparsity                  :", 1 - nnz / (n_user * n_item))
print("Memori sparse (MB)        :", (M_user.data.nbytes + M_user.indices.nbytes + M_user.indptr.nbytes) / 1e6)
print("Memori jika dense (GB)    :", n_user * n_item * 4 / 1e9)

item_per_user = np.diff(M_user.indptr)     # jumlah item berbeda per user
print("\nItem berbeda per user:")
print(pd.Series(item_per_user).describe())
print("User dengan hanya 1 item  :", (item_per_user == 1).sum(),
      f"({(item_per_user == 1).mean():.1%})")
print("User dengan >= 5 item     :", (item_per_user >= 5).sum())

In [ ]:
# Sel 2.1.5c - Siapkan matriks transpose (Item x User) dan derajat tiap item.
# Dibuat di sini karena plot no. 4 memakai derajat item; dipakai ulang pada no. 5-8 dan 2.2.
M_item = M_user.T.tocsr()                  # baris = item, kolom = user
user_per_item = np.diff(M_item.indptr)     # jumlah user berbeda yang menyentuh tiap item

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

rng = np.random.default_rng(42)          # seed tetap -> bisa direproduksi

# 1) cari item "jembatan" yang dipakai 6-12 user, lalu ambil user yang itemnya tidak terlalu banyak
for _ in range(100):
    bridge_cands = np.where((user_per_item >= 6) & (user_per_item <= 12))[0]
    bridge = int(rng.choice(bridge_cands))
    users_b = [u for u in M_item.getrow(bridge).indices if 3 <= item_per_user[u] <= 8]
    if len(users_b) >= 4:
        break

sampel_u = list(rng.choice(users_b, size=min(6, len(users_b)), replace=False))

# 2) bangun graf bipartit sampel
G = nx.Graph()
for u in sampel_u:
    a, b = M_user.indptr[u], M_user.indptr[u + 1]
    for j, w in zip(M_user.indices[a:b], M_user.data[a:b]):
        G.add_edge(("U", int(user_ids[u])), ("I", int(item_ids[j])), weight=float(w), u=u)

user_nodes = [("U", int(user_ids[u])) for u in sampel_u]
item_nodes = [n for n in G if n[0] == "I"]

# 3) posisi: user di kiri (rata), item di kanan diurutkan menurut rata-rata posisi user terhubung
y_user = {n: -i for i, n in enumerate(user_nodes)}
bary = {n: np.mean([y_user[v] for v in G[n]]) for n in item_nodes}
item_sorted = sorted(item_nodes, key=lambda n: -bary[n])
y_item = np.linspace(0, -(len(user_nodes) - 1), len(item_sorted))

pos = {n: (0, y_user[n]) for n in user_nodes}
pos.update({n: (1, y) for n, y in zip(item_sorted, y_item)})

# 4) gambar
warna = plt.cm.tab10(np.arange(len(user_nodes)))
warna_user = dict(zip(user_nodes, warna))
shared = [n for n in item_nodes if G.degree(n) >= 2]
unik = [n for n in item_nodes if G.degree(n) == 1]

fig, ax = plt.subplots(figsize=(13, 8))

for (a, b, d) in G.edges(data=True):
    un = a if a[0] == "U" else b
    ax.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]],
            color=warna_user[un], alpha=0.55, linewidth=0.6 + 0.5 * d["weight"], zorder=1)

nx.draw_networkx_nodes(G, pos, nodelist=user_nodes, node_color=list(warna_user.values()),
                       node_size=700, edgecolors="black", ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=unik, node_color="lightgray", node_size=70,
                       edgecolors="gray", ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=shared, node_color="red", node_size=160,
                       edgecolors="black", ax=ax)

# label user di sisi kiri, label item di sisi kanan (tidak menimpa node)
for n in user_nodes:
    ax.text(-0.04, pos[n][1], f"User {n[1]}", ha="right", va="center", fontsize=10, fontweight="bold")
for n in item_sorted:
    ax.text(1.02, pos[n][1], f"Item {n[1]}", ha="left", va="center", fontsize=8,
            color="red" if n in shared else "dimgray")

ax.set_xlim(-0.35, 1.25)
ax.set_title("Cuplikan graf bipartit User–Item (sampel reproducible, seed=42)\n"
             "Tebal garis = bobot interaksi | Merah = item yang dipakai ≥ 2 user", fontsize=12)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markeredgecolor="black", markersize=12, label="User"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markeredgecolor="black", markersize=9, label="Item dipakai ≥ 2 user"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray", markeredgecolor="gray", markersize=7, label="Item dipakai 1 user"),
], loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)
ax.axis("off")
plt.tight_layout()
plt.show()

print("Jumlah user:", len(user_nodes), "| item:", len(item_nodes), "| item bersama:", len(shared), "| edge:", G.number_of_edges())

**Karakteristik VSM User**

| Karakteristik | Nilai |
|---|---|
| Shape matriks User × Item | (870.345, 190.242) |
| Dimensi tiap vektor user | 190.242 (satu dimensi per item) |
| Elemen non-nol | 1.311.993 pasangan (user, item) unik |
| Kepadatan / sparsity | 7,92 × 10⁻⁶ / 99,9992% |
| Memori sparse (CSR) | sekitar 14 MB |
| Memori jika dense | sekitar 662 GB (sekitar 47 ribu kali lebih besar) |

- **Sangat sparse.** Hampir seluruh sel bernilai nol, sehingga matriks wajib disimpan dalam format sparse. Elemen non-nol (1.311.993) lebih sedikit dari jumlah relasi (1.683.395) karena interaksi berulang pada pasangan visitor–item yang sama digabung menjadi satu skor.
- **Vektor user sangat pendek isinya.** Rata-rata user hanya menyentuh 1,5 item berbeda (median 1, maksimum 2.956). Sebanyak **698.712 user (80,3%) hanya berinteraksi dengan satu item**, dan hanya 23.713 user (sekitar 2,7%) yang menyentuh minimal 5 item.
- **Plot networkx** menampilkan sampel 6 user acak (`seed=42`; jumlah user, item, item bersama, dan edge tercetak di bawah plot). Graf bipartit terlihat jelas: tiap user (warna berbeda) terhubung ke beberapa item, dan item yang dipakai lebih dari satu user (merah) menjadi "jembatan" antar user, sedangkan sebagian besar item hanya dipakai satu user (abu-abu). Ini menggambarkan sparsity di atas.

### No. 5 — VSM simpul Item

Sekarang setiap **Item** dipandang sebagai vektor pada ruang berdimensi sebanyak jumlah **User**: tiap dimensi mewakili satu user dan nilainya adalah skor interaksi user terhadap item itu. Matriksnya **Item × User** (`M_item`), yaitu transpose dari `M_user` (informasi yang sama dibaca dari sisi lain).

In [ ]:
# Sel 5.1: matriks Item x User
M_item = M_user.T.tocsr()

n_item, n_user = M_item.shape
nnz = M_item.nnz
user_per_item = np.diff(M_item.indptr)   # jumlah user berbeda per item

print("Shape matriks Item x User :", M_item.shape)
print("Dimensi vektor item       :", n_user, "(satu dimensi per user)")
print("Jumlah elemen non-nol     :", nnz)
print("Kepadatan (density)       :", nnz / (n_item * n_user))
print("Sparsity                  :", 1 - nnz / (n_item * n_user))
print("Memori sparse (MB)        :", (M_item.data.nbytes + M_item.indices.nbytes + M_item.indptr.nbytes) / 1e6)

print("\nUser berbeda per item:")
print(pd.Series(user_per_item).describe())
print("Item dengan hanya 1 user  :", (user_per_item == 1).sum(),
      f"({(user_per_item == 1).mean():.1%})")
print("Item dengan >= 5 user     :", (user_per_item >= 5).sum())

In [ ]:
# Sel 5.2: plot networkx untuk sisi Item
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

rng = np.random.default_rng(42)          # seed tetap -> bisa direproduksi

# 1) cari user "jembatan" yang menyentuh 6-12 item, lalu ambil item yang pengunjungnya tidak terlalu banyak
for _ in range(100):
    bridge_cands = np.where((item_per_user >= 6) & (item_per_user <= 12))[0]
    bridge = int(rng.choice(bridge_cands))
    items_b = [i for i in M_user.getrow(bridge).indices if 3 <= user_per_item[i] <= 8]
    if len(items_b) >= 4:
        break

sampel_i = list(rng.choice(items_b, size=min(6, len(items_b)), replace=False))

# 2) bangun graf bipartit sampel
G_item = nx.Graph()
for i in sampel_i:
    a, b = M_item.indptr[i], M_item.indptr[i + 1]
    for j, w in zip(M_item.indices[a:b], M_item.data[a:b]):
        G_item.add_edge(("I", int(item_ids[i])), ("U", int(user_ids[j])), weight=float(w))

item_nodes = [("I", int(item_ids[i])) for i in sampel_i]
user_nodes = [n for n in G_item if n[0] == "U"]

# 3) posisi: item di kiri (rata), user di kanan diurutkan menurut rata-rata posisi item terhubung
y_item = {n: -k for k, n in enumerate(item_nodes)}
bary = {n: np.mean([y_item[v] for v in G_item[n]]) for n in user_nodes}
user_sorted = sorted(user_nodes, key=lambda n: -bary[n])
y_user = np.linspace(0, -(len(item_nodes) - 1), len(user_sorted))

pos = {n: (0, y_item[n]) for n in item_nodes}
pos.update({n: (1, y) for n, y in zip(user_sorted, y_user)})

# 4) gambar
warna_item = dict(zip(item_nodes, plt.cm.tab10(np.arange(len(item_nodes)))))
shared = [n for n in user_nodes if G_item.degree(n) >= 2]
unik = [n for n in user_nodes if G_item.degree(n) == 1]

fig, ax = plt.subplots(figsize=(13, 8))

for (a, b, d) in G_item.edges(data=True):
    itn = a if a[0] == "I" else b
    ax.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]],
            color=warna_item[itn], alpha=0.55, linewidth=0.6 + 0.5 * d["weight"], zorder=1)

nx.draw_networkx_nodes(G_item, pos, nodelist=item_nodes, node_color=list(warna_item.values()),
                       node_size=700, edgecolors="black", ax=ax)
nx.draw_networkx_nodes(G_item, pos, nodelist=unik, node_color="lightgray", node_size=70,
                       edgecolors="gray", ax=ax)
nx.draw_networkx_nodes(G_item, pos, nodelist=shared, node_color="red", node_size=160,
                       edgecolors="black", ax=ax)

# label item di kiri, label user di kanan (tidak menimpa node)
for n in item_nodes:
    ax.text(-0.04, pos[n][1], f"Item {n[1]}", ha="right", va="center", fontsize=10, fontweight="bold")
for n in user_sorted:
    ax.text(1.02, pos[n][1], f"User {n[1]}", ha="left", va="center", fontsize=8,
            color="red" if n in shared else "dimgray")

ax.set_xlim(-0.35, 1.25)
ax.set_title("Cuplikan graf bipartit Item–User (sampel reproducible, seed=42)\n"
             "Tebal garis = bobot interaksi | Merah = user yang menyentuh ≥ 2 item", fontsize=12)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markeredgecolor="black", markersize=12, label="Item"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markeredgecolor="black", markersize=9, label="User menyentuh ≥ 2 item"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray", markeredgecolor="gray", markersize=7, label="User menyentuh 1 item"),
], loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)
ax.axis("off")
plt.tight_layout()
plt.show()

print("Jumlah item:", len(item_nodes), "| user:", len(user_nodes), "| user bersama:", len(shared), "| edge:", G_item.number_of_edges())

**Karakteristik VSM Item**

| Karakteristik | Nilai |
|---|---|
| Shape matriks Item × User | (190.242, 870.345) |
| Dimensi tiap vektor item | 870.345 (satu dimensi per user) |
| Elemen non-nol | 1.311.993 (sama dengan matriks User × Item) |
| Kepadatan / sparsity | 7,92 × 10⁻⁶ / 99,9992% |
| Memori sparse (CSR) | sekitar 11 MB |

- Matriks ini **tetap sangat sparse**, tetapi vektor item lebih "terisi" dibanding vektor user: rata-rata satu item disentuh **6,9 user berbeda** (median 2, maksimum 1.792).
- Hanya **73.187 item (38,5%) yang dipakai satu user**, jauh lebih kecil daripada 80,3% pada sisi user. Sebanyak 57.626 item (sekitar 30%) dipakai minimal 5 user.
- **Plot networkx** menampilkan sampel 5 item acak (`seed=42`; jumlah item, user, user bersama, dan edge tercetak di bawah plot) dengan item di kiri dan user yang menyentuhnya di kanan. Dua item dapat dianggap mirip bila dikunjungi user yang sama; user yang menyentuh lebih dari satu item (merah) menjadi penghubung antar item.

### No. 6 — Perbandingan proyeksi: User-User vs Item-Item

Dari matriks bipartit `M` (User × Item) ada dua proyeksi graf:
- **User-User** = `M · Mᵀ`: dua user terhubung bila menyentuh item yang sama.
- **Item-Item** = `Mᵀ · M`: dua item terhubung bila disentuh user yang sama.

Perbandingan dilakukan dari sisi **efisiensi** (ukuran matriks hasil dan estimasi operasi, yaitu $\sum \text{derajat}^2$) dan **efektivitas** (seberapa banyak vektor yang punya cukup informasi untuk dibandingkan).

In [ ]:
# Sel 6.1: ukuran dan estimasi biaya proyeksi
n_user, n_item = M_user.shape

# estimasi operasi = jumlah pasangan yang berbagi tetangga (sum derajat^2)
kerja_user_proj = float((user_per_item.astype(np.float64) ** 2).sum())   # lewat item -> pasangan user
kerja_item_proj = float((item_per_user.astype(np.float64) ** 2).sum())   # lewat user -> pasangan item

perbandingan = pd.DataFrame({
    "Proyeksi": ["User-User (M·Mᵀ)", "Item-Item (Mᵀ·M)"],
    "Jumlah simpul": [n_user, n_item],
    "Ukuran matriks hasil": [f"{n_user} x {n_user}", f"{n_item} x {n_item}"],
    "Sel maksimum": [float(n_user) ** 2, float(n_item) ** 2],
    "Estimasi operasi (sum derajat^2)": [kerja_user_proj, kerja_item_proj],
    "Vektor dgn 1 interaksi": [
        f"{(item_per_user == 1).mean():.1%}",
        f"{(user_per_item == 1).mean():.1%}",
    ],
})
display(perbandingan)
print("Rasio biaya User/Item :", kerja_user_proj / kerja_item_proj)

In [ ]:
# Sel 6.2: ukur nyata pada sampel kecil (waktu dan jumlah elemen non-nol)
import time

rng = np.random.default_rng(42)
sub_u = rng.choice(n_user, size=5000, replace=False)
sub_i = rng.choice(n_item, size=5000, replace=False)

A = M_user[sub_u, :]          # 5000 user x semua item
B = M_item[sub_i, :]          # 5000 item x semua user

t0 = time.time(); S_user = A @ A.T; t_user = time.time() - t0
t0 = time.time(); S_item = B @ B.T; t_item = time.time() - t0

print(f"User-User (5000 user): nnz={S_user.nnz}, waktu={t_user:.3f}s")
print(f"Item-Item (5000 item): nnz={S_item.nnz}, waktu={t_item:.3f}s")

# diagonal = 5.000 elemen (tiap simpul dengan dirinya sendiri); sisanya pasangan simpul yang saling terhubung
print("Pasangan terhubung di luar diagonal -> User-User:", S_user.nnz - 5000,
      "| Item-Item:", S_item.nnz - 5000)

**Hasil perbandingan**

| Ukuran | User-User | Item-Item |
|---|---|---|
| Jumlah simpul | 870.345 | 190.242 |
| Sel maksimum matriks hasil | 7,6 × 10¹¹ | 3,6 × 10¹⁰ (sekitar 21× lebih kecil) |
| Estimasi operasi ($\sum$ derajat²) | 75,3 juta | 38,5 juta (sekitar 2× lebih ringan) |
| Vektor yang hanya punya 1 interaksi | 80,3% | 38,5% |
| Rata-rata interaksi per vektor | 1,5 item per user | 6,9 user per item |
| Sampel 5.000 simpul: pasangan terhubung di luar diagonal | jauh lebih sedikit | jauh lebih banyak (lihat output Sel 6.2) |
| Sampel 5.000 simpul: waktu hitung | sangat singkat | sangat singkat |

**Kesimpulan: proyeksi Item-Item lebih baik**, dengan alasan:
1. **Lebih efisien:** matriks hasil sekitar 21 kali lebih kecil dan estimasi operasi sekitar 2 kali lebih sedikit.
2. **Lebih efektif:** vektor item lebih kaya informasi. Pada sisi user, 80,3% vektor hanya punya satu elemen non-nol sehingga hampir tidak punya tetangga untuk dibandingkan, sedangkan pada sisi item hanya 38,5%.
3. **Lebih banyak pasangan yang saling terhubung** pada sampel berukuran sama (lihat output Sel 6.2, nilai `nnz` dikurangi 5.000 elemen diagonal). Catatan: 5.000 user hanya sekitar 0,57% populasi user, sedangkan 5.000 item sekitar 2,6% populasi item, sehingga angka sampel ini tidak sepenuhnya sebanding dan dipakai sebagai pendukung saja.
4. Waktu hitung pada sampel kecil sama-sama sangat singkat, sehingga keunggulan Item-Item terletak pada ukuran dan kualitas informasi, bukan kecepatan.

Karena itu **no. 7 dan analitik 2.2 memakai proyeksi Item-Item** (similarity antar-item, rekomendasi berbasis item).

### No. 7 — Perhitungan similarity pada graf terproyeksi (Item-Item)

Similarity antar-item dihitung dengan **cosine similarity** pada vektor item (baris `M_item`). Tiap baris dinormalisasi dengan norma L2, sehingga perkalian titik dua baris sama dengan cosine-nya (0 sampai 1).

Menulis seluruh pasangan item ke graf tidak realistis (puluhan juta relasi), sehingga:
1. similarity dihitung dari matriks sparse untuk **satu item acuan** (soal: record acuan ditentukan sendiri). Dipilih **item 215800**, yang awalnya diambil acak (`seed=42`) dari item yang disentuh minimal 20 user, lalu dikunci agar hasil, query, dan screenshot konsisten;
2. **5 item paling mirip** ditulis balik ke graf sebagai relasi `(:Item)-[:SIMILAR_TO {cosine}]->(:Item)` (soal 7b);
3. query Cypher mengambil kembali 5 record dengan similarity tertinggi (soal 7a).

Urutan di notebook: hitung, tulis ke graf (b), lalu query (a), karena query membaca nilai yang sudah ada di graf.

In [ ]:
# Pastikan tidak ada relasi SIMILAR_TO sisa dari proses sebelumnya,
# supaya kondisi "SEBELUM" similarity selalu 0 (reproducible saat Run All).
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.execute_query("MATCH ()-[s:SIMILAR_TO]->() DELETE s", database_="neo4j")

In [ ]:
# Sel 7.0: kondisi SEBELUM similarity (untuk no. 8)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    rec, _, _ = driver.execute_query(
        "MATCH ()-[s:SIMILAR_TO]->() RETURN count(s) AS jumlah",
        database_="neo4j", routing_=RoutingControl.READ)
print("Jumlah relasi SIMILAR_TO sebelum:", rec[0]["jumlah"])

In [ ]:
# Sel 7.1: hitung cosine similarity item acuan terhadap semua item
M_item_norm = normalize(M_item, norm="l2", axis=1)     # tiap baris dinormalisasi -> dot product = cosine

ref_item_id = 215800                                   # item acuan (ditentukan sendiri, dikunci)
ref_idx = int(np.where(item_ids == ref_item_id)[0][0])
assert user_per_item[ref_idx] >= 20, "item acuan harus disentuh minimal 20 user"

sim = (M_item_norm @ M_item_norm[ref_idx].T).toarray().ravel()
sim[ref_idx] = -1                                      # buang item itu sendiri
top5_idx = np.argsort(-sim, kind="stable")[:5]         # stable: seri diurutkan menurut ID item

print("Item acuan:", ref_item_id, "| jumlah user:", user_per_item[ref_idx])
hasil = pd.DataFrame({
    "item_mirip": [int(item_ids[j]) for j in top5_idx],
    "cosine": [float(sim[j]) for j in top5_idx],
    "jumlah_user": [int(user_per_item[j]) for j in top5_idx],
})
display(hasil)

In [ ]:
# Sel 7.2: (b) tulis nilai similarity ke graf
rows = [{"ref": ref_item_id, "other": int(item_ids[j]), "sim": float(sim[j])} for j in top5_idx]

q_write = """
UNWIND $rows AS r
MATCH (a:Item {itemId: r.ref}), (b:Item {itemId: r.other})
MERGE (a)-[s:SIMILAR_TO]->(b)
SET s.cosine = r.sim
"""

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.execute_query(q_write, rows=rows, database_="neo4j")
print("Relasi SIMILAR_TO ditulis:", len(rows))

In [ ]:
# Sel 7.3: (a) CYPHER untuk 5 record dengan similarity tertinggi
q_top5 = """
MATCH (a:Item {itemId: $ref})-[s:SIMILAR_TO]->(b:Item)
RETURN a.itemId AS item_acuan, b.itemId AS item_mirip, s.cosine AS cosine
ORDER BY s.cosine DESC, b.itemId ASC   // seri: ID item lebih kecil dulu
LIMIT 5
"""
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    rec, _, _ = driver.execute_query(q_top5, ref=ref_item_id,
                                     database_="neo4j", routing_=RoutingControl.READ)
display(pd.DataFrame([r.data() for r in rec]))

**Query Cypher untuk 5 similarity tertinggi (7a)**
```cypher
MATCH (a:Item {itemId: $ref})-[s:SIMILAR_TO]->(b:Item)
RETURN a.itemId AS item_acuan, b.itemId AS item_mirip, s.cosine AS cosine
ORDER BY s.cosine DESC
LIMIT 5
```
- `MATCH (a:Item {itemId: $ref})-[s:SIMILAR_TO]->(b:Item)` mencari item acuan beserta item-item yang terhubung lewat relasi `SIMILAR_TO`; `$ref` adalah parameter yang diisi dari Python.
- `ORDER BY s.cosine DESC LIMIT 5` mengurutkan dari similarity tertinggi dan mengambil 5 teratas.

**Hasil.** Item acuan adalah **215800** (disentuh 26 user). Lima item paling mirip:

| Peringkat | Item mirip | Cosine | Jumlah user |
|---|---|---|---|
| 1 | 241950 | 0,5281 | 15 |
| 2 | 129918 | 0,5244 | 13 |
| 3 | 401111 | 0,5244 | 4 |
| 4 | 144878 | 0,5132 | 4 |
| 5 | 250224 | 0,4509 | 15 |

Nilai top-5 pada Sel 7.1 (dihitung dari matriks) sama dengan hasil Sel 7.3 (dibaca kembali dari graf Neo4j), sehingga penulisan ke graf benar. Item 129918 dan 401111 memiliki cosine yang sama (seri), sehingga urutan keduanya dapat tertukar. Nilai cosine yang tinggi pada item dengan hanya 4 user menunjukkan similarity pada data sparse bisa dipengaruhi jumlah user yang kecil.

### No. 8 — Kondisi graf sebelum dan sesudah perhitungan similarity

In [ ]:
# Sel 8.1: hitung sesudah (bandingkan dengan Sel 7.0)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    rec, _, _ = driver.execute_query(
        "MATCH ()-[s:SIMILAR_TO]->() RETURN count(s) AS jumlah",
        database_="neo4j", routing_=RoutingControl.READ)
print("Jumlah relasi SIMILAR_TO sesudah:", rec[0]["jumlah"])
print("Item acuan untuk query Browser  :", ref_item_id)

| Kondisi | Jumlah relasi `SIMILAR_TO` |
|---|---|
| Sebelum similarity | 0 |
| Sesudah similarity | 5 |

**Query Cypher untuk tampilan graf (dijalankan di Neo4j Browser)**

*Sebelum* — lingkungan item acuan hanya berisi relasi interaksi dari Visitor (tanpa `SIMILAR_TO`); `LIMIT` dipakai agar tampilan tidak berat:
```cypher
MATCH (a:Item {itemId: 215800})<-[r]-(v:Visitor)
RETURN a, r, v LIMIT 25
```

*Sesudah* — item acuan terhubung ke 5 item paling mirip dengan nilai `cosine` pada relasinya:
```cypher
MATCH p = (a:Item {itemId: 215800})-[:SIMILAR_TO]->(b:Item)
RETURN p
```
**Tampilan graf sebelum:**

![Graf sebelum similarity](images/graf_sebelum.png)

**Tampilan graf sesudah:**

![Graf sesudah similarity](images/graf_sesudah.png)

## 2.2 Analytics

### Rancangan metode rekomendasi (KNN berbasis item)

Sesuai kesimpulan no. 6, rekomendasi memakai **proyeksi Item-Item**. Untuk satu user target:
1. Ambil semua item yang disentuh user pada **P1** beserta bobot interaksinya (1/3/5).
2. Untuk setiap item tersebut, cari **k = 5 tetangga terdekat** (cosine tertinggi, item itu sendiri dibuang).
3. Skor tiap kandidat = jumlah dari (bobot item user × cosine) lintas semua item user.
4. Buang item yang sudah pernah disentuh user di P1, urutkan skor menurun (seri: ID item lebih kecil dulu), ambil top-5.
5. Jika kandidat kurang dari 5, lengkapi dengan item terpopuler yang belum pernah dilihat user.

**Pemilihan user acak (reproducible).** Dari user yang ada di P1 dan P2, kandidat dibatasi pada user dengan minimal 3 item berbeda di P1 dan minimal 1 item **baru** di P2 (belum pernah disentuh di P1). Satu user dipilih dengan `np.random.default_rng(42)` dari daftar kandidat yang terurut menurut ID.

**Ground truth** = item baru yang disentuh user tersebut di **P2**.

In [ ]:
#siapkan ground truth P2 dan fungsi bantu
df_p2_eval = pd.read_csv("data/events_p2_test.csv", dtype={"transactionid": "string"})

item_to_idx = {int(i): k for k, i in enumerate(item_ids)}
user_to_idx = {int(u): k for k, u in enumerate(user_ids)}

def user_items(k_user):
    """indeks item & bobot interaksi user di P1"""
    a, b = M_user.indptr[k_user], M_user.indptr[k_user + 1]
    return M_user.indices[a:b], M_user.data[a:b]

# item yang disentuh tiap user di P2
p2_by_user = df_p2_eval.groupby("visitorid")["itemid"].apply(lambda s: set(s.astype(int)))
print("User di P2:", len(p2_by_user))

In [ ]:
# Pilih 1 user secara acak
kandidat_user = []
for u_id in sorted(set(p2_by_user.index.astype(int)) & set(user_to_idx)):
    k = user_to_idx[u_id]
    if item_per_user[k] < 3:
        continue
    seen = set(int(item_ids[j]) for j in user_items(k)[0])
    relevan = {i for i in p2_by_user[u_id] if i not in seen and i in item_to_idx}
    if relevan:
        kandidat_user.append(u_id)

print("Jumlah user kandidat:", len(kandidat_user))

rng = np.random.default_rng(42)                 # seed tetap -> bisa direproduksi
target_user_id = int(rng.choice(kandidat_user))
target_k = user_to_idx[target_user_id]
print("User terpilih:", target_user_id, "| item di P1:", item_per_user[target_k])

In [ ]:
#rekomendasi knn
def recommend_item_knn(k_user, k=5, topn=5):
    idx, w = user_items(k_user)
    seen = set(int(j) for j in idx)

    # similarity tiap item user ke semua item (sparse), lalu ambil k tetangga terdekat
    S = (M_item_norm[idx] @ M_item_norm.T).tocsr()
    skor = {}
    for r, j in enumerate(idx):
        a, b = S.indptr[r], S.indptr[r + 1]
        cols, vals = S.indices[a:b], S.data[a:b]
        m = cols != j                              # buang item itu sendiri
        cols, vals = cols[m], vals[m]
        if len(vals) == 0:
            continue
        top = np.argsort(-vals, kind="stable")[:k]   # k tetangga terdekat (seri: indeks/ID lebih kecil dulu)
        for c, v in zip(cols[top], vals[top]):
            skor[int(c)] = skor.get(int(c), 0.0) + float(w[r]) * float(v)

    # buang item yang sudah pernah disentuh user, urutkan skor (seri -> itemId kecil dulu)
    kandidat = [(c, s) for c, s in skor.items() if c not in seen]
    kandidat.sort(key=lambda x: (-x[1], int(item_ids[x[0]])))
    rekom = [int(item_ids[c]) for c, _ in kandidat[:topn]]
    skor_rekom = [s for _, s in kandidat[:topn]]

    # jika kurang dari topn, lengkapi dengan item terpopuler yang belum dilihat user
    if len(rekom) < topn:
        populer = np.argsort(-user_per_item)
        for c in populer:
            if int(c) not in seen and int(item_ids[c]) not in rekom:
                rekom.append(int(item_ids[c])); skor_rekom.append(0.0)
            if len(rekom) == topn:
                break
    return rekom, skor_rekom

rekom5, skor5 = recommend_item_knn(target_k, k=5, topn=5)

In [ ]:
# 3 Item yang diiklankan
iklan3 = rekom5[:3]
print("User:", target_user_id)
print("Item yang sudah disentuh di P1:", [int(item_ids[j]) for j in user_items(target_k)[0]][:20])
print("\nTop-5 rekomendasi (k=5):")
display(pd.DataFrame({"rank": range(1, 6), "itemid": rekom5, "skor": skor5}))
print("3 item yang diiklankan:", iklan3)

**Hasil (no. 1b).** User terpilih adalah **127311** (menyentuh 33 item berbeda di P1; dipilih dari 3.347 user kandidat). Top-5 rekomendasi dengan k = 5 adalah 51448, 434571, 464783, 63846, dan 338063. **Tiga item yang diiklankan: 51448, 434571, dan 464783.** Item 51448 dan 434571 memiliki skor yang sama (1,9103), sehingga urutannya ditentukan oleh ID item yang lebih kecil.

### No. 2 — Evaluasi: Hit@5 dan NDCG@5

Evaluasi memakai top-5 rekomendasi dan ground truth (item baru di P2).

In [ ]:
# Hit@5 dan NDCG@5
seen_ids = set(int(item_ids[j]) for j in user_items(target_k)[0])
relevan = {i for i in p2_by_user[target_user_id] if i not in seen_ids and i in item_to_idx}
print("Ground truth (item baru di P2):", relevan)

def hit_at_k(rekom, relevan, k=5):
    return int(any(i in relevan for i in rekom[:k]))

def ndcg_at_k(rekom, relevan, k=5):
    dcg = sum(1 / np.log2(r + 1) for r, i in enumerate(rekom[:k], start=1) if i in relevan)
    idcg = sum(1 / np.log2(r + 1) for r in range(1, min(len(relevan), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0

print("Hit@5  :", hit_at_k(rekom5, relevan, 5))
print("NDCG@5 :", round(ndcg_at_k(rekom5, relevan, 5), 4))

In [ ]:
# evaluasi rata-rata pada banyak user
rng = np.random.default_rng(42)
sampel = rng.choice(kandidat_user, size=min(300, len(kandidat_user)), replace=False)

hits, ndcgs = [], []
for u_id in sampel:
    k = user_to_idx[int(u_id)]
    r5, _ = recommend_item_knn(k, k=5, topn=5)
    seen_u = set(int(item_ids[j]) for j in user_items(k)[0])
    rel_u = {i for i in p2_by_user[int(u_id)] if i not in seen_u and i in item_to_idx}
    hits.append(hit_at_k(r5, rel_u, 5))
    ndcgs.append(ndcg_at_k(r5, rel_u, 5))

print(f"Rata-rata Hit@5  ({len(sampel)} user): {np.mean(hits):.4f}")
print(f"Rata-rata NDCG@5 ({len(sampel)} user): {np.mean(ndcgs):.4f}")

**Hasil dan interpretasi (no. 2)**

| Evaluasi | Hit@5 | NDCG@5 |
|---|---|---|
| User 127311 (ground truth: 16206, 205275, 49446) | 0 | 0,0000 |
| Rata-rata 300 user acak (seed=42) | 0,0100 | 0,0051 |

- Untuk user 127311, tidak ada satu pun dari 5 item rekomendasi (51448, 434571, 464783, 63846, 338063) yang cocok dengan 3 item baru yang disentuhnya di P2, sehingga **Hit@5 = 0** dan **NDCG@5 = 0**.
- Pada 300 user acak, rata-rata Hit@5 hanya 0,01 (sekitar 3 dari 300 user), jadi hasil user 127311 bukan kasus khusus, melainkan mewakili kinerja umum model.
- **Penyebab nilai rendah:**
  1. Data sangat sparse (lihat no. 3–5), sehingga tetangga item hanya didukung sedikit user dan cosine mudah dipengaruhi kebetulan.
  2. Ground truth hanya memuat item **baru** (belum pernah disentuh di P1), yang sulit ditebak dari riwayat; banyak di antaranya mungkin item dengan sedikit atau tanpa interaksi di P1 (*cold-start*).
  3. Ruang kandidat sangat besar (190.242 item) sedangkan rekomendasi hanya 5 item, sehingga peluang tepat sangat kecil.
  4. Interaksi didominasi `view`, sedangkan sinyal minat kuat (`BOUGHT`) hanya 0,82%.
- Pengembangan yang mungkin: memakai bobot popularitas, memperpanjang periode training, atau mengombinasikan similarity item dengan sinyal lain.